# Projekt: Jobbannons-filter
**Namn:** Elias Paleovrachas Haag
**Kurs:** Utveckling med Python, grund
**Verktyg:** VS Code, Jobtech API, Git/GitHub

### Projektbeskrivning
Detta projekt bygger en automatiserad datapipeline som hämtar realtidsdata om jobbannonser för programmeringsspråk från Arbetsförmedlingens **JobTech API**. Datamängden valideras och tvättas med objektorienterade principer (OOP) och sparas säkert som strukturerad rådata (CSV) samt renad data med GDPR-metadata (JSON).

# 1. Importer och konfiguration
Importerar standardbibliotek och sätter upp listan med godkända programmeringsspråk.

In [36]:
import os
from datetime import datetime
import requests
import json



accepted_languages = ["Python", "Java", "C++", "C#", "SQL"]

# 2. Objektorientering och Arv (OOP)
Skapar förälderklassen `Jobb_annons` med valideringsregler samt barnklassen `Distans_jobb` som ärver dess egenskaper.


In [37]:
class Jobb_annons:
    def __init__(self, language, location, date_created):
        self.language = language
        self.location = location
        self.date_created = date_created
        
#Koppling till yrkesrollen: Precis som Lovable gör, hjälper denna funktion att ta emot de värde som användaren ger och sedan gå igenom den godkända listan för att säkerställa att inga felaktiga värden når AI-modellen.
    def validate_language(self):
      return self.language in accepted_languages
        
    def validate_location(self):
      if not self.location or self.location in ["None", "Okänd ort"]:
         return False
      return True

class Distans_jobb(Jobb_annons):
   def __init__(self, language, location, distans, date_created):
      super().__init__(language, location, date_created)
      self.distans = distans      





# 3. API-hämtning och datatvätt
Hämtar data från JobTech API, skapar objekt och tvättar datan med klassens valideringsmetoder.

In [38]:
os.makedirs("output", exist_ok=True)

clean_data_list = []
kasserade_rader = 0
tidsstampel = datetime.now().strftime("%Y-%m-%d %H:%M")

print("Anropar JobTech API för alla språk...")

for sprak in accepted_languages:
  url = f"https://jobsearch.api.jobtechdev.se/search?q={sprak}&limit=5"

  try:
    response = requests.get(url)

    if response.status_code == 200:
      api_data = response.json()
      annonser = api_data.get("hits", [])

      for annons in annonser:
        ort = annons.get("workplace_address", {}).get(
            "municipality", "Okänd ort"
        )
        is_remote = annons.get("workplace_address", {}).get(
            "workplace_remote_approved", False
        )

        ny_annons_objekt = Distans_jobb(
            language=sprak,
            location=ort,
            distans=is_remote,
            date_created=tidsstampel,
        )

        if (
            ny_annons_objekt.validate_language()
            and ny_annons_objekt.validate_location()
        ):
          clean_data_list.append({
              "language": ny_annons_objekt.language,
              "location": ny_annons_objekt.location,
              "is_remote": ny_annons_objekt.distans,
              "date_created": ny_annons_objekt.date_created,
          })
        else:
          kasserade_rader += 1

      print(f"Hämtade jobb för {sprak}")
    else:
      print(f"Kunde inte hämta data för {sprak}. Felkod: {response.status_code}")

  except Exception as e:
    print(f"Ett nätverksfel uppstod för {sprak}: {e}")

print(
    f"\nGodkända rader: {len(clean_data_list)}. Kasserade rader:"
    f" {kasserade_rader}"
)




Anropar JobTech API för alla språk...
Hämtade jobb för Python
Hämtade jobb för Java
Hämtade jobb för C++
Hämtade jobb för C#
Hämtade jobb för SQL

Godkända rader: 28. Kasserade rader: 2


# 4. Dataexport med GDPR-metadata
Exporterar den godkända datan till en JSON-fil tillsammans med källa, tidsstämpel och GDPR-status.


In [39]:
if clean_data_list:
  json_output = {
      "source": "Jobtech API (Arbetsförmedlingen)",
      "gdpr_compliant": True,
      "created": tidsstampel,
      "data": clean_data_list,
  }

  json_file_path = os.path.join("output", "clean_data.json")
  with open(json_file_path, "w", encoding="utf-8") as json_fil:
    json.dump(json_output, json_fil, ensure_ascii=False, indent=4)

  print(f"Renad data har sparats i: {json_file_path}")
    

        

Renad data har sparats i: output\clean_data.json


## 5. Etik & Framtidsarkitektur

### EU AI Act (Riskanalys)
* **Minimal risk:** Analyserar offentliga jobbannonser – ingen automatisk rekrytering sker [1].
* **Datakvalitet:** Datatvätten rensar felaktiga värden så att framtida AI-modeller får korrekt underlag [1].

### Multi-Agent System (MAS)
Vid en MAS-arkitektur skulle koden delas upp på två agenter:
1. **Data-agent:** Hämtar rådata från API [2].
2. **Analys-agent:** Validerar, tvättar och exporterar till JSON med GDPR-metadata [2].
